# Дообучение EEG-DINO на NMT → проверка на турецком наборе

| Поле | Значение |
|------|----------|
| **Автор** | Захар Санников (zs) |
| **Дата** | 2026-10-22 |
| **Гипотеза** | Г1 (ЭЭГ) |
| **Цель** | Дообучить EEG-DINO (MICCAI 2025, DOI 10.1007/978-3-032-04927-8_19, код и веса HF `eegdino/EEG-DINO`) по той же схеме, что REVE, CBraMod, BIOT и LUNA, и сравнить / собрать в ансамбль с REVE |
| **Веса** | `pre-trained-models/model_EEG_DINO_M.pt` (снимок репозитория `5d8ae132`, 28.02.2025). Общие предобученные веса, **не** детектор разрядов SenuaLab (тот — EEG-DINO Medium, дообученная на vEpiSet). В файле «студент» и «учитель» DINO + проекторы; как в `run_finetuning.py` авторов, берётся студент (`module.student.*` → кодировщик), загрузка строгая |
| **Предобучение** | По статье — **только TUEG** (TUH EEG Corpus, >30 000 записей, >16 000 пациентов), подготовка как у CBraMod: 19 каналов 10-20, 200 Гц, куски 30 с, 1 109 545 кусков, >9000 ч. Исключение пациентов TUAB/TUEV и других подмножеств не упомянуто. **Риск утечки для TUEP**: пациенты TUEP почти наверняка в предобучении — при проверке на TUEP это нужно учитывать (как у CBraMod, REVE, LUNA) |
| **Выбор варианта** | Medium (33 млн параметров, 512/16/1024). У авторов на TUAB при полном дообучении S 0,898 / M 0,902 / L 0,910 — прирост M → L всего +0,008 при ×6 вычислений (201 млн): Large с выбором скорости и 3 запусками в 1,5 ч T4 не укладывается. Small дешевле, но M по всем задачам авторов не хуже S и укладывается в бюджет |
| **Вход** | Выяснено по коду и статье: 200 Гц, патч = 1 с = 200 отсчётов (`rfft` на 101 частоту, `rearrange(... T=200)`), вход в мкВ/100 (`samples / 100` в `engine_finetuning.py`), **19 монополярных каналов**: вложение канала — one-hot по номеру канала (19 классов), поэтому порядок каналов важен; берём порядок подготовки TUEG у CBraMod (Fp1 Fp2 F3 F4 C3 C4 P3 P4 O1 O2 F7 F8 T3 T4 T5 T6 Fz Cz Pz), на который ссылаются авторы. Окна 10 с (как TUAB у авторов и как REVE/BIOT у нас) → 19 × 10 патчей. Фильтр 0,3–75 Гц + режекция 50 Гц, как у CBraMod |
| **Амплитуда** | Как у CBraMod в наших блокнотах: каждое окно приводится к общему СКО 15 мкВ по всем каналам (абсолютная амплитуда пропадает, соотношение каналов остаётся), затем /100 как у авторов. Масштаб NMT в файлах неверный — так он не важен |
| **Аугментации** | как у CBraMod: шум 0,1·СКО, зеркальная замена полушарий (p = 0,5), зануление 1–2 каналов (p = 0,5), сдвиг до 2 с |
| **Обучение** | полное дообучение (вся модель + голова авторов из `run_finetuning.py` на 1 выход), BCE, AdamW (wd 0,05, без wd для смещений и норм), разогрев 10 % + косинус, пакет 64, fp16 (вложение патчей с БПФ — в fp32). Послойное снижение скорости 0,9 (параметр авторов; в их коде оно фактически не срабатывает — имена слоёв не совпадают с `blocks.*`). Базовая скорость — по отложенным 15 % NMT train из 5·10⁻⁵ / 1,5·10⁻⁴ / 5·10⁻⁴ (5·10⁻⁴ — значение авторов при пакете 512). Эпохи — до выхода на плато: до 10, остановка, если отложенный ROC AUC не растёт 2 эпохи подряд; берётся лучшая эпоха. Затем 3 запуска (третий не уложился в бюджет и досчитан отдельным блокнотом — раздел 6а) |
| **Среда** | Kaggle, GPU T4 |

In [ ]:
import os, sys, glob, time, types, warnings
ON_KAGGLE = os.path.exists("/kaggle/input")
SMOKE = False   # True — короткая проверка на процессоре (часть записей, 1 эпоха), тестовые цифры не смотреть
HF_REPO, HF_REV = "eegdino/EEG-DINO", "5d8ae13252a4170789140b10e3e91fd32f603155"   # снимок репозитория авторов, проверен 21.10.2026
SIZE = "S" if SMOKE and not ON_KAGGLE else "M"
warnings.filterwarnings("ignore", category=FutureWarning)
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F, scipy.io, scipy.signal as ss
from sklearn.metrics import roc_auc_score, roc_curve
from scipy.stats import rankdata, spearmanr
from concurrent.futures import ThreadPoolExecutor
DEV = "cuda" if torch.cuda.is_available() else "cpu"
OUT = "/kaggle/working" if ON_KAGGLE else "."
T_NB = time.time()

SFREQ, SEG, PATCH = 200, 10, 200
L = SFREQ * SEG
CAP_TRAIN, CAP_EVAL = 8, 60   # окон по 10 с на запись, как у REVE и BIOT

# порядок каналов подготовки TUEG у CBraMod = номера one-hot в channel_embedding EEG-DINO
EL = ["Fp1", "Fp2", "F3", "F4", "C3", "C4", "P3", "P4", "O1", "O2", "F7", "F8", "T3", "T4", "T5", "T6", "Fz", "Cz", "Pz"]
MIRROR_PAIRS = {"Fp1": "Fp2", "F3": "F4", "C3": "C4", "P3": "P4", "O1": "O2", "F7": "F8", "T3": "T4", "T5": "T6"}
MIRROR_PAIRS.update({v: k for k, v in list(MIRROR_PAIRS.items())})
MIRROR = [EL.index(MIRROR_PAIRS.get(e, e)) for e in EL]

CFG = {"S": dict(feature_size=200, num_layers=12, dim_feedforward=512, emb="embedding_small"),
       "M": dict(feature_size=512, num_layers=16, dim_feedforward=1024, emb="embedding_medium"),
       "L": dict(feature_size=1024, num_layers=24, dim_feedforward=2048, emb="embedding_large")}[SIZE]
NUM_HEADS, NUM_GLOBAL, GLOBAL_LAYER = 8, 1, 1   # значения по умолчанию run_finetuning.py (число голов в статье не указано)

if ON_KAGGLE:
    from huggingface_hub import hf_hub_download
    get = lambda f: hf_hub_download(HF_REPO, f, revision=HF_REV, cache_dir="/kaggle/temp/hf")   # не в /kaggle/working: веса не попадают в вывод
else:
    get = lambda f: os.path.join(os.environ["DINO_DIR"], f)

def load_src(name, path, fixes=()):
    """Модуль авторов из файла с точечными правками (каждая правка должна найтись ровно один раз)."""
    src = open(path, encoding="utf-8").read()
    for a, b in fixes:
        assert src.count(a) == 1, (name, a)
        src = src.replace(a, b)
    mod = types.ModuleType(name); sys.modules[name] = mod
    exec(compile(src, path, "exec"), mod.__dict__)
    return mod

# .cuda() зашит в код авторов — заменяем на устройство входа (нужно для проверки на процессоре), больше ничего не меняем
emb = load_src("dino_embedding", get(f"models/{CFG['emb']}.py"),
               [("torch.arange(self.num_channels+1).cuda()", "torch.arange(self.num_channels+1, device=x.device)")])
trf = load_src("dino_transformer", get("models/transformer.py"))
_pe_forward = emb.PatchEmbedding.forward
def _pe_forward_fp32(self, x):   # rfft длины 200 в fp16 на GPU не работает (длина не степень двойки) — вложение патчей в fp32
    with torch.autocast("cuda", enabled=False):
        return _pe_forward(self, x.float())
emb.PatchEmbedding.forward = _pe_forward_fp32

class EEGEncoder(nn.Module):
    """Как models/eeg_encoder.py авторов (там лишний импорт monai и жёстко embedding_small)."""
    def __init__(self):
        super().__init__()
        d = CFG["feature_size"]
        self.patch_embedding = emb.PatchEmbedding(d_model=d)
        self.encoder_layers = nn.ModuleList([trf.TransformerEncoderLayer(d_model=d, nhead=NUM_HEADS, dim_feedforward=CFG["dim_feedforward"])
                                             for _ in range(CFG["num_layers"])])
        self.global_tokens = nn.Parameter(torch.randn(1, NUM_GLOBAL, d))
    def forward(self, x_in):
        x = self.patch_embedding(x_in)            # [B, C, P, D]
        b = x.shape[0]
        x = x.reshape(b, -1, x.shape[-1])         # [B, C*P, D]
        g = self.global_tokens.expand(b, -1, -1)
        for i, layer in enumerate(self.encoder_layers):
            x = layer(x)
            if i + 1 == GLOBAL_LAYER:
                x = torch.cat([g, x], dim=1)
        return x

class DinoNet(nn.Module):
    """Кодировщик + голова ClassificationModel из run_finetuning.py (среднее по каналам, затем по времени, MLP) на 1 выход."""
    def __init__(self):
        super().__init__()
        d = CFG["feature_size"]
        self.encoder = EEGEncoder()
        self.full_linear, self.channel_linear = nn.Linear(d, d), nn.Linear(d, d)
        self.classifier = nn.Sequential(nn.Linear(d, d // 2), nn.GELU(), nn.Dropout(0.5),
                                        nn.Linear(d // 2, d // 4), nn.GELU(), nn.Dropout(0.3), nn.Linear(d // 4, 1))
    def forward(self, x):
        bs, ch, P, _ = x.shape
        f = self.encoder(x)[:, NUM_GLOBAL:]
        f = F.gelu(self.full_linear(f)).reshape(bs, ch, P, -1).mean(1)   # [bs, P, d]
        f = F.gelu(self.channel_linear(f)).mean(1)                       # [bs, d]
        return self.classifier(f).squeeze(-1)

CK = torch.load(get(f"pre-trained-models/model_EEG_DINO_{SIZE}.pt"), map_location="cpu", weights_only=True)["state_dict"]
STUDENT = {k[len("module.student."):]: v for k, v in CK.items() if k.startswith("module.student.")}
print("в файле весов:", {g: sum(k.startswith(f"module.{g}.") for k in CK) for g in sorted({k.split(".")[1] for k in CK})})

def new_dino(seed):
    torch.manual_seed(seed)
    m = DinoNet()
    m.encoder.load_state_dict(STUDENT, strict=True)   # строго: ни одного пропущенного или лишнего ключа кодировщика
    return m.to(DEV)

_m = new_dino(0)
_enc = set(_m.encoder.state_dict()); _head = [n for n in _m.state_dict() if not n.startswith("encoder.")]
assert _enc == set(STUDENT), "ключи кодировщика и студента не совпали"
print(f"EEG-DINO-{SIZE}: кодировщик {len(_enc)} тензоров загружено строго, без обучения остаются только тензоры головы ({len(_head)}): {sorted({n.split('.')[0] for n in _head})}")
print("параметров:", sum(p.numel() for p in _m.parameters()), "| из них кодировщик:", sum(p.numel() for p in _m.encoder.parameters()))
del _m

CB_STD_UV, NOISE = 15.0, 0.1
def batch_input(W, g=None):
    """(B, 19, 2000) -> окно к общему СКО 15 мкВ -> мкВ/100 как у авторов -> (аугментации) -> (B, 19, 10, 200)."""
    x = torch.from_numpy(np.ascontiguousarray(W)).to(DEV).float()
    x = x - x.mean(-1, keepdim=True)
    x = (x / (x.std(dim=(1, 2), keepdim=True) + 1e-6) * (CB_STD_UV / 100.0)).clamp(-5, 5)   # обрезка ±33 СКО — только выбросы
    if g is not None:
        x = x + NOISE * (CB_STD_UV / 100.0) * torch.randn_like(x)
        m = torch.from_numpy(g.random(len(x)) < 0.5).to(DEV)
        x[m] = x[m][:, MIRROR]
        for b in range(len(x)):
            if g.random() < 0.5:
                x[b, g.choice(19, g.integers(1, 3), replace=False)] = 0
            x[b] = torch.roll(x[b], int(g.integers(-2 * SFREQ, 2 * SFREQ + 1)), dims=-1)
    return x.reshape(len(x), 19, SEG, PATCH)

AMP = dict(device_type="cuda", dtype=torch.float16, enabled=DEV == "cuda")
def predict(model, Ws, bs=128):
    """Медиана вероятности по окнам для каждой записи."""
    model.eval(); out = []
    with torch.no_grad(), torch.autocast(**AMP):
        for W in Ws:
            p = [torch.sigmoid(model(batch_input(W[b:b + bs])).float()).cpu().numpy() for b in range(0, len(W), bs)]
            out.append(np.median(np.concatenate(p)))
    return np.array(out)

def rank(s): return rankdata(s) / len(s)
def boot_ci(yt, s, n=2000, seed=0):
    r = np.random.default_rng(seed); v = []
    for _ in range(n):
        i = r.integers(0, len(yt), len(yt))
        if len(set(yt[i])) == 2: v.append(roc_auc_score(yt[i], s[i]))
    return np.percentile(v, [2.5, 97.5])

SOS_BP = ss.butter(4, [0.3, 75], btype="bandpass", fs=SFREQ, output="sos")
B_N, A_N = ss.iirnotch(50, 30, fs=SFREQ)
def cb_filter(X):
    """0,3-75 Гц + режекция 50 Гц, как в блокнотах CBraMod, REVE и BIOT."""
    if X.shape[-1] < 60 * SFREQ:
        return X
    return ss.filtfilt(B_N, A_N, ss.sosfiltfilt(SOS_BP, X.astype(np.float64), axis=-1), axis=-1)

def windows(X):
    n = X.shape[1] // L
    return X[:, :n * L].reshape(19, n, L).transpose(1, 0, 2)

def spread(n, k):
    return np.unique(np.linspace(0, n - 1, k).astype(int)) if n > k else np.arange(n)
print("устройство:", torch.cuda.get_device_name(0) if DEV == "cuda" else "процессор")

## 2. NMT и турецкий набор (подкачка обоих параллельно)

In [ ]:
import mne
mne.set_log_level("ERROR")
ROOT_NMT = glob.glob("/kaggle/input/**/nmt_scalp_eeg_dataset", recursive=True)[0]
ROOT_TR = glob.glob("/kaggle/input/**/Turkish Epilepsy Dataset", recursive=True)[0]
lab = pd.read_csv(f"{ROOT_NMT}/Labels.csv")
paths = {os.path.basename(p): p for p in glob.glob(f"{ROOT_NMT}/**/*.edf", recursive=True)}
lab["path"] = lab["recordname"].map(paths)
lab = lab.dropna(subset=["path"]).sort_values("recordname").reset_index(drop=True)
lab["y"] = (lab["label"].str.lower() == "abnormal").astype(int)
tfiles = [(f, 1) for f in sorted(glob.glob(f"{ROOT_TR}/epilepsy/*.mat"))] + \
         [(f, 0) for f in sorted(glob.glob(f"{ROOT_TR}/normal/*.mat"))]
if SMOKE:
    lab = pd.concat([lab[(lab["loc"] == "train") & (lab.y == 0)].head(40), lab[(lab["loc"] == "train") & (lab.y == 1)].head(20),
                     lab[(lab["loc"] == "eval")].head(20)]).sort_values("recordname").reset_index(drop=True)
    tfiles = tfiles[:10] + tfiles[-10:]
print(pd.crosstab(lab["loc"], lab["label"]))

def warm(p):
    with open(p, "rb") as f:
        while f.read(1 << 24):
            pass
t0 = time.time(); allf = list(lab.path) + [f for f, _ in tfiles]
with ThreadPoolExecutor(32) as ex:
    for i, _ in enumerate(ex.map(warm, allf)):
        if (i + 1) % 500 == 0 or i + 1 == len(allf):
            print(f"подкачано {i+1}/{len(allf)} {time.time()-t0:.0f} с", flush=True)

ALIAS = {"T7": "T3", "T8": "T4", "P7": "T5", "P8": "T6"}
TARGET = [e.upper() for e in EL]
def norm(ch):
    c = ch.upper().replace("EEG", "").strip()
    for s in ["-REF", "-AVG", "-LE", "-AR"]:
        c = c.replace(s, "")
    return ALIAS.get(c.strip(" -_"), c.strip(" -_"))

NMT_STORE_K = 100.0   # масштаб в файлах неверный; окно всё равно нормируется, множитель только для float16
def nmt_job(i):
    r = lab.iloc[i]
    try:
        raw = mne.io.read_raw_edf(r.path, preload=True)
        if raw.info["sfreq"] != SFREQ:
            raw.resample(SFREQ)
        names = {norm(c): c for c in raw.ch_names}
        X = raw.get_data(picks=[names[t] for t in TARGET]) * 1e6 * NMT_STORE_K
    except Exception:
        return r.recordname, None
    W = windows(cb_filter(X))
    if len(W) == 0:
        return r.recordname, None
    if r["loc"] == "train" and len(W) > CAP_TRAIN:
        W = W[np.sort(np.random.default_rng(i).choice(len(W), CAP_TRAIN, replace=False))]
    elif r["loc"] == "eval":
        W = W[spread(len(W), CAP_EVAL)]
    return r.recordname, W.astype(np.float16)

NMT = {}
t0 = time.time()
with ThreadPoolExecutor(4) as ex:
    for i, (name, res) in enumerate(ex.map(nmt_job, range(len(lab)))):
        if res is not None:
            NMT[name] = res
        if (i + 1) % 400 == 0 or i + 1 == len(lab):
            print(f"NMT прочитано {i+1}/{len(lab)} {time.time()-t0:.0f} с", flush=True)
lab = lab[lab.recordname.isin(NMT)].reset_index(drop=True)
tr, ev = lab[lab["loc"] == "train"].reset_index(drop=True), lab[lab["loc"] == "eval"].reset_index(drop=True)
print(pd.crosstab(lab["loc"], lab["label"]))
print("окон по 10 с: train", sum(len(NMT[n]) for n in tr.recordname), "| eval", sum(len(NMT[n]) for n in ev.recordname))

## 3. Турецкий набор: первые 19 каналов (относительно уха) → средний референт, 500→200 Гц, куски по 15 с подряд, окна 10 с

In [ ]:
# индексы турецких каналов в порядке EL: Fp1 Fp2 F3 F4 C3 C4 P3 P4 O1 O2 F7 F8 T3 T4 T5 T6 Fz Cz Pz
IDX = [0, 1, 2, 3, 5, 6, 8, 9, 11, 12, 13, 14, 15, 16, 17, 18, 4, 7, 10]

def tr_job(f):
    d = scipy.io.loadmat(f)["data"]
    X = np.stack([np.stack([d[s, c].ravel() for c in IDX]) for s in range(d.shape[0])])   # (куски, 19, 7500)
    X = X - X.mean(1, keepdims=True)          # средний референт, как в блокноте CBraMod
    X = ss.resample_poly(X, 2, 5, axis=-1)    # 200 Гц
    W = windows(cb_filter(np.concatenate(list(X), axis=-1)))
    return W[spread(len(W), CAP_EVAL)].astype(np.float16), float(np.std(X))

TR, TR_AMP = [], []
t0 = time.time()
with ThreadPoolExecutor(4) as ex:
    for a, b in ex.map(tr_job, [f for f, _ in tfiles]):
        TR.append(a); TR_AMP.append(b)
y_tr = np.array([yy for _, yy in tfiles]); tr_ids = [os.path.basename(f) for f, _ in tfiles]
TR_AMP = np.array(TR_AMP)
print(f"Турция прочитано за {time.time()-t0:.0f} с | эпилепсия {int(y_tr.sum())}, здоровые {int((1-y_tr).sum())} | окон на запись: медиана {int(np.median([len(w) for w in TR]))}")
print("эталон перекоса: СКО сигнала записи как оценка, ROC AUC =", round(roc_auc_score(y_tr, TR_AMP), 3))

## 4. Дообучение EEG-DINO

In [ ]:
from sklearn.model_selection import train_test_split

fit_names, val_names = train_test_split(tr.recordname.to_numpy(), test_size=0.15, stratify=tr.y, random_state=0)
ymap = dict(zip(lab.recordname, lab.y))
yw = np.concatenate([np.full(len(NMT[n]), ymap[n]) for n in fit_names])
y_val = np.array([ymap[n] for n in val_names])
Xw = np.concatenate([NMT[n] for n in fit_names])
val_W = [NMT[n] for n in val_names]
print(f"окон для обучения: {len(yw)} (патология {yw.mean():.2f}) | записей на выбор: {len(val_names)}")

BS, MAX_EP, PATIENCE, LLRD, WD = 64, (1 if SMOKE else 10), 2, 0.9, 0.05
w_cls = np.where(yw == 1, 0.5 / (yw == 1).sum(), 0.5 / (yw == 0).sum()); w_cls /= w_cls.sum()

def layer_scale(n):
    """Послойное снижение: слой i кодировщика -> 0,9^(16 - i), вложение патчей и глобальный токен -> 0,9^17, голова -> 1."""
    nl = CFG["num_layers"]
    if n.startswith("encoder.encoder_layers."):
        return LLRD ** (nl - int(n.split(".")[2]))
    return LLRD ** (nl + 1) if n.startswith("encoder.") else 1.0

def make_opt(model, total_steps, lr):
    groups = {}
    for n, p in model.named_parameters():
        groups.setdefault((layer_scale(n), p.ndim > 1), []).append(p)   # смещения и нормы — без weight decay (как create_optimizer в timm)
    opt = torch.optim.AdamW([{"params": ps, "lr": lr * k, "weight_decay": WD if dec else 0.0} for (k, dec), ps in groups.items()], betas=(0.9, 0.999))
    warm_n = max(1, int(0.1 * total_steps))
    f = lambda s: (s + 1) / warm_n if s < warm_n else 0.5 * (1 + np.cos(np.pi * (s - warm_n) / max(1, total_steps - warm_n))) * 0.99 + 0.01
    return opt, torch.optim.lr_scheduler.LambdaLR(opt, f)

def step(model, opt, sched, scaler, x, t):
    with torch.autocast(**AMP):
        out = model(x)
    loss = F.binary_cross_entropy_with_logits(out.float(), t.float())
    bad = not torch.isfinite(loss)
    opt.zero_grad()
    if not bad:
        scaler.scale(loss).backward(); scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update()
    sched.step()
    return loss.item(), bad

def train_run(seed, lr, tag="dino", deadline=None):
    """До MAX_EP эпох; остановка, если отложенный ROC AUC не растёт PATIENCE эпох подряд (плато) или кончается время. Возвращается лучшая эпоха."""
    g = np.random.default_rng(seed)
    model = new_dino(seed)
    steps_ep = int(np.ceil(len(yw) / BS))
    opt, sched = make_opt(model, MAX_EP * steps_ep, lr)
    scaler = torch.amp.GradScaler("cuda", enabled=DEV == "cuda")
    best_auc, best_state, best_ep, hist, n_bad, stop = -1, None, 0, [], 0, "макс. эпох"
    t0 = time.time()
    for ep in range(MAX_EP):
        model.train(); losses = []
        idx = g.choice(len(yw), size=len(yw), replace=True, p=w_cls)
        for b in range(0, len(idx), BS):
            j = idx[b:b + BS]
            l, bad = step(model, opt, sched, scaler, batch_input(Xw[j], g), torch.tensor(yw[j], device=DEV))
            n_bad += bad; losses.append(l)
        auc = roc_auc_score(y_val, predict(model, val_W))
        hist.append((round(float(np.nanmean(losses)), 3), round(float(auc), 3)))
        print(f"  {tag} lr {lr:g} запуск {seed} | эпоха {ep+1}: потери {np.nanmean(losses):.3f} | отложенные NMT {auc:.3f} | шагов с NaN/inf {n_bad} | {time.time()-t0:.0f} с", flush=True)
        if auc > best_auc:
            best_auc, best_ep, best_state = auc, ep + 1, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        if ep + 1 - best_ep >= PATIENCE:
            stop = "плато"; break
        if deadline is not None and time.time() + (time.time() - t0) / (ep + 1) > deadline:
            stop = "время"; break
    print(f"  {tag} lr {lr:g} запуск {seed}: остановка — {stop}, лучшая эпоха {best_ep}", flush=True)
    model.load_state_dict(best_state)
    return model, best_ep, best_auc, hist, n_bad, stop

In [ ]:
# --- проверка сходимости на малой подвыборке (без аугментаций): потери должны убывать и уйти ниже 0,69 ---
g0 = np.random.default_rng(123)
n_sm, ep_sm = (32, 3) if SMOKE else (128, 8)
sm = np.concatenate([g0.choice(np.where(yw == c)[0], n_sm, replace=False) for c in (0, 1)])
model = new_dino(0); opt, sched = make_opt(model, ep_sm * (len(sm) // BS), 1.5e-4)
scaler = torch.amp.GradScaler("cuda", enabled=DEV == "cuda")
model.train(); sm_losses, tt = [], []
for ep in range(ep_sm):
    g0.shuffle(sm); ls = []
    for b in range(0, len(sm), BS):
        j = np.sort(sm[b:b + BS]); t1 = time.time()
        l, bad = step(model, opt, sched, scaler, batch_input(Xw[j]), torch.tensor(yw[j], device=DEV))
        if DEV == "cuda": torch.cuda.synchronize()
        tt.append(time.time() - t1); ls.append(l)
    sm_losses.append(np.mean(ls)); print(f"  проверка: эпоха {ep+1} потери {sm_losses[-1]:.3f}", flush=True)
t_step = float(np.median(tt[2:])) if len(tt) > 3 else float(np.median(tt))
del model, opt
if DEV == "cuda": torch.cuda.empty_cache()
CONVERGED = bool(np.isfinite(sm_losses).all() and np.mean(sm_losses[-2:]) < np.mean(sm_losses[:2]) - 0.05 and min(sm_losses) < 0.66)
print("сходимость на подвыборке:", "есть" if CONVERGED else f"НЕТ ({np.round(sm_losses, 3)}) — продолжаю, скорость выбирается по отложенным 15 %")
est_ep = ((len(yw) / BS) * t_step * 1.3 + 15) / 60   # мин на эпоху (+30 % на аугментации, ~15 с на отложенные)
print(f"шаг {t_step:.3f} с | оценка одной эпохи {est_ep:.1f} мин")

In [ ]:
# --- выбор базовой скорости обучения: только по отложенным 15 % NMT train (запуск 0) ---
TIME_LIMIT_MIN = 1 if SMOKE else 58   # бюджет задачи 1,5 ч GPU; чтение данных ~15-20 мин
T_TRAIN, GRID, dur = time.time(), {}, []
DEADLINE = T_TRAIN + TIME_LIMIT_MIN * 60
left = lambda: (DEADLINE - time.time()) / 60
for i, lr in enumerate([1.5e-4] if SMOKE else [1.5e-4, 5e-5, 5e-4]):
    est = np.mean(dur) / 60 if dur else 6 * est_ep
    if i >= 2 and left() < est * 3:   # третья скорость — только если после неё хватит времени на 2 запуска
        print(f"ПРОПУСК lr {lr:g}: не укладывается (осталось {left():.0f} мин)"); continue
    t0 = time.time()
    model, best_ep, best_auc, hist, n_bad, stop = train_run(0, lr, tag="выбор", deadline=DEADLINE - 2 * est * 60)
    dur.append(time.time() - t0)
    GRID[lr] = dict(model=model.cpu(), best_ep=best_ep, best_auc=best_auc, hist=hist, n_bad=n_bad, stop=stop)
    print(f"lr {lr:g}: лучшая эпоха {best_ep}, отложенные NMT {best_auc:.3f}, {dur[-1]/60:.1f} мин", flush=True)
LR = max(GRID, key=lambda k: GRID[k]["best_auc"])
print("выбрана скорость:", LR)
pd.DataFrame([dict(lr=k, best_epoch=v["best_ep"], val=round(v["best_auc"], 3), stop=v["stop"], hist=v["hist"], nan_steps=v["n_bad"]) for k, v in GRID.items()]) \
  .to_csv(f"{OUT}/eegdino_lr_choice.csv", index=False)
for k in [k for k in GRID if k != LR]: del GRID[k]["model"]

# --- 3 запуска с выбранной скоростью (запуск 0 — уже обученный при выборе); тестовые наборы — один раз ---
RUNS = []
for seed in ([0] if SMOKE else [0, 1, 2]):
    t0 = time.time()
    if seed == 0:
        G = GRID[LR]; model, best_ep, best_auc, hist, n_bad, stop = G["model"].to(DEV), G["best_ep"], G["best_auc"], G["hist"], G["n_bad"], G["stop"]
    else:
        if left() < 0.5 * np.mean(dur) / 60:
            print(f"ПРОПУСК запуск {seed}: не укладывается в {TIME_LIMIT_MIN} мин"); continue
        model, best_ep, best_auc, hist, n_bad, stop = train_run(seed, LR, deadline=DEADLINE)
    s_ev = predict(model, [NMT[n] for n in ev.recordname]); s_tr = predict(model, TR)
    RUNS.append(dict(model="eegdino", lr=LR, seed=seed, best_epoch=best_ep, stop=stop, val=round(best_auc, 3), hist=hist, nan_steps=n_bad,
                     nmt_eval=roc_auc_score(ev.y, s_ev), turkey=roc_auc_score(y_tr, s_tr), s_ev=s_ev, s_tr=s_tr,
                     turkey_med_p0=float(np.median(s_tr[y_tr == 0])), turkey_med_p1=float(np.median(s_tr[y_tr == 1])),
                     n_unique_tr=len(np.unique(s_tr)), s_val=predict(model, val_W)))
    print(f"eegdino запуск {seed}: эпоха {best_ep} | NMT eval {RUNS[-1]['nmt_eval']:.3f} | Турция {RUNS[-1]['turkey']:.3f} | "
          f"медиана p здоровые/эпилепсия {RUNS[-1]['turkey_med_p0']:.3f}/{RUNS[-1]['turkey_med_p1']:.3f} | "
          f"разных значений {RUNS[-1]['n_unique_tr']}/{len(s_tr)} | {(time.time()-t0)/60:.1f} мин", flush=True)
    del model
    if DEV == "cuda": torch.cuda.empty_cache()
del GRID
print(f"всё обучение: {(time.time()-T_TRAIN)/60:.1f} мин")
runs = pd.DataFrame([{k: v for k, v in r.items() if k not in ("s_ev", "s_tr", "s_val")} for r in RUNS])
runs.to_csv(f"{OUT}/eegdino_finetune_runs.csv", index=False)
print(runs.drop(columns="hist").round(3).to_string(index=False))

## 5. Итоги: ROC AUC, пороговые метрики (порог — по отложенным 15 % NMT train, на Турцию перенесён тот же)

In [ ]:
y_ev = ev.y.to_numpy()
ens_ev = np.mean([rank(r["s_ev"]) for r in RUNS], 0); ens_tr = np.mean([rank(r["s_tr"]) for r in RUNS], 0)
def fmt(y, s):
    lo, hi = boot_ci(y, s); return f"{roc_auc_score(y, s):.3f} ({lo:.2f}-{hi:.2f})"
sd = lambda v: v.std(ddof=1) if len(v) > 1 else 0.0
summary = pd.DataFrame([
    {"модель": f"EEG-DINO-{SIZE} дообученная, запуски", "n": len(runs), "NMT eval": f"{runs.nmt_eval.mean():.3f} ± {sd(runs.nmt_eval):.3f}",
     "Турция": f"{runs.turkey.mean():.3f} ± {sd(runs.turkey):.3f} ({' / '.join(f'{v:.2f}' for v in runs.turkey)})"},
    {"модель": f"EEG-DINO-{SIZE} дообученная, ансамбль запусков", "n": "", "NMT eval": fmt(y_ev, ens_ev), "Турция": fmt(y_tr, ens_tr)}])
summary.to_csv(f"{OUT}/eegdino_finetune_summary.csv", index=False)
print(summary.to_string(index=False))
print("Связь с амплитудой записи на Турции (Спирмен):", round(spearmanr(ens_tr, TR_AMP).correlation, 2))

# пороговые метрики: оценка — среднее вероятностей запусков (у рангов нет общего масштаба между наборами)
p_val = np.mean([r["s_val"] for r in RUNS], 0); p_ev = np.mean([r["s_ev"] for r in RUNS], 0); p_tr = np.mean([r["s_tr"] for r in RUNS], 0)
fpr, tpr, thr = roc_curve(y_val, p_val)
THR = {"Юден": float(thr[np.argmax(tpr - fpr)]), "чувствительность 80 %": float(thr[np.argmax(tpr >= 0.8)])}
def metr(y, pr):
    tp, tn = ((pr == 1) & (y == 1)).sum(), ((pr == 0) & (y == 0)).sum(); fp, fn = ((pr == 1) & (y == 0)).sum(), ((pr == 0) & (y == 1)).sum()
    rec = tp / max(tp + fn, 1); spec = tn / max(tn + fp, 1)
    return dict(acc=(tp + tn) / len(y), bacc=(rec + spec) / 2, prec=tp / (tp + fp) if tp + fp else np.nan, rec=rec, spec=spec)
def metr_ci(y, s, t, n=2000, seed=0):
    pr = (s >= t).astype(int); m0 = metr(y, pr); r = np.random.default_rng(seed); B = []
    for _ in range(n):
        i = r.integers(0, len(y), len(y)); B.append(metr(y[i], pr[i]))
    return {k: f"{m0[k]:.2f} ({np.nanpercentile([b[k] for b in B], 2.5):.2f}-{np.nanpercentile([b[k] for b in B], 97.5):.2f})" for k in m0}
rows = []
for name, t in THR.items():
    for ds, y, s in [("NMT eval", y_ev, p_ev), ("Турция", y_tr, p_tr)]:
        m = metr_ci(y, s, t)
        rows.append({"модель": "EEG-DINO", "порог": name, "значение порога": round(t, 3), "набор": ds,
                     "примечание": "порог: отложенные 15 % NMT train" + ("" if ds == "NMT eval" else "; перенесён"), "n": len(y),
                     "accuracy": m["acc"], "сбаланс. accuracy": m["bacc"], "precision": m["prec"], "recall (чувств.)": m["rec"], "специфичность": m["spec"]})
thr_tab = pd.DataFrame(rows)
thr_tab.to_csv(f"{OUT}/eegdino_threshold_metrics.csv", index=False)
print(thr_tab.drop(columns=["примечание"]).to_string(index=False))

d1 = pd.DataFrame({"dataset": "nmt_eval", "id": ev.recordname.tolist(), "y": y_ev}); d2 = pd.DataFrame({"dataset": "turkey", "id": tr_ids, "y": y_tr, "amp": TR_AMP})
for d, k in [(d1, "s_ev"), (d2, "s_tr")]:
    for r in RUNS: d[f"eegdino_seed{r['seed']}"] = r[k]
d1["eegdino_ens_rank"], d2["eegdino_ens_rank"] = ens_ev, ens_tr
d3 = pd.DataFrame({"dataset": "nmt_val", "id": list(val_names), "y": y_val})
for r in RUNS: d3[f"eegdino_seed{r['seed']}"] = r["s_val"]
pd.concat([d1, d2, d3]).to_csv(f"{OUT}/eegdino_finetune_per_person.csv", index=False)
print(f"\nвсё время блокнота: {(time.time()-T_NB)/60:.1f} мин")

## 6. Вывод Kaggle (без строк о подкачке файлов и служебных предупреждений)

Тестовые наборы считались один раз. NaN/inf нет, потери на всём NMT train падают (у выбранной скорости 0,66–0,67 → 0,44–0,46), у групп вероятности различны (119 разных значений из 121). Перед этим блокнот прошёл короткую проверку на процессоре Kaggle (`SMOKE = True`, отдельный блокнот `zakharsann/h1-eegdino-smoke`: 60 записей NMT train, 20 eval, 20 турецких, 1 эпоха, 9,4 мин, GPU не тратился); её тестовые цифры для решений не использовались.

Проверка сходимости на подвыборке (256 окон, 8 эпох) формально «НЕТ»: потери упали 0,694 → 0,66, а критерий (взят у LUNA) требует падения на 0,05 за 8 эпох с разогревом. На полном обучении модель учится нормально, так что это не мешает.

**Здесь запусков 2**: третий (seed 2) не уложился в лимит обучения 58 мин (бюджет 1,5 ч GPU). Причина: выбор скорости занял 35 мин (3 значения по 10–12 мин), а второй запуск дошёл до 8 эпох. Третий досчитан отдельно — раздел 6а; итоги ниже уже по 3 запускам.

```
в файле весов: {'crops_projector': 8, 'dino_loss': 1, 'ibot_patch_loss': 1, 'mask_projector': 8, 'student': 227, 'teacher': 227}
EEG-DINO-M: кодировщик 227 тензоров загружено строго, без обучения остаются только тензоры головы (10): ['channel_linear', 'classifier', 'full_linear']
параметров: 34445121 | из них кодировщик: 33755456
устройство: Tesla T4
label  abnormal  normal
loc                    
eval         90      95
train       325    1907
NMT прочитано 400/2417 146 с
NMT прочитано 800/2417 287 с
NMT прочитано 1200/2417 427 с
NMT прочитано 1600/2417 577 с
NMT прочитано 2000/2417 721 с
NMT прочитано 2400/2417 878 с
NMT прочитано 2417/2417 886 с
label  abnormal  normal
loc                    
eval         90      95
train       325    1904
окон по 10 с: train 17775 | eval 10406
Турция прочитано за 122 с | эпилепсия 50, здоровые 71 | окон на запись: медиана 60
эталон перекоса: СКО сигнала записи как оценка, ROC AUC = 0.764
окон для обучения: 15103 (патология 0.14) | записей на выбор: 335
  проверка: эпоха 1 потери 0.694
  проверка: эпоха 2 потери 0.693
  проверка: эпоха 3 потери 0.688
  проверка: эпоха 4 потери 0.679
  проверка: эпоха 5 потери 0.675
  проверка: эпоха 6 потери 0.658
  проверка: эпоха 7 потери 0.663
  проверка: эпоха 8 потери 0.665
сходимость на подвыборке: НЕТ ([0.694 0.693 0.688 0.679 0.675 0.658 0.663 0.665]) — продолжаю, скорость выбирается по отложенным 15 %
шаг 0.420 с | оценка одной эпохи 2.4 мин
  выбор lr 0.00015 запуск 0 | эпоха 1: потери 0.652 | отложенные NMT 0.811 | шагов с NaN/inf 0 | 112 с
  выбор lr 0.00015 запуск 0 | эпоха 2: потери 0.565 | отложенные NMT 0.856 | шагов с NaN/inf 0 | 230 с
  выбор lr 0.00015 запуск 0 | эпоха 3: потери 0.504 | отложенные NMT 0.871 | шагов с NaN/inf 0 | 352 с
  выбор lr 0.00015 запуск 0 | эпоха 4: потери 0.439 | отложенные NMT 0.877 | шагов с NaN/inf 0 | 474 с
  выбор lr 0.00015 запуск 0 | эпоха 5: потери 0.379 | отложенные NMT 0.876 | шагов с NaN/inf 0 | 596 с
  выбор lr 0.00015 запуск 0 | эпоха 6: потери 0.335 | отложенные NMT 0.873 | шагов с NaN/inf 0 | 720 с
  выбор lr 0.00015 запуск 0: остановка — плато, лучшая эпоха 4
lr 0.00015: лучшая эпоха 4, отложенные NMT 0.877, 12.0 мин
  выбор lr 5e-05 запуск 0 | эпоха 1: потери 0.663 | отложенные NMT 0.756 | шагов с NaN/inf 0 | 125 с
  выбор lr 5e-05 запуск 0 | эпоха 2: потери 0.602 | отложенные NMT 0.852 | шагов с NaN/inf 0 | 251 с
  выбор lr 5e-05 запуск 0 | эпоха 3: потери 0.523 | отложенные NMT 0.872 | шагов с NaN/inf 0 | 377 с
  выбор lr 5e-05 запуск 0 | эпоха 4: потери 0.464 | отложенные NMT 0.842 | шагов с NaN/inf 0 | 502 с
  выбор lr 5e-05 запуск 0 | эпоха 5: потери 0.411 | отложенные NMT 0.862 | шагов с NaN/inf 0 | 629 с
  выбор lr 5e-05 запуск 0: остановка — плато, лучшая эпоха 3
lr 5e-05: лучшая эпоха 3, отложенные NMT 0.872, 10.5 мин
  выбор lr 0.0005 запуск 0 | эпоха 1: потери 0.660 | отложенные NMT 0.768 | шагов с NaN/inf 0 | 125 с
  выбор lr 0.0005 запуск 0 | эпоха 2: потери 0.634 | отложенные NMT 0.789 | шагов с NaN/inf 0 | 248 с
  выбор lr 0.0005 запуск 0 | эпоха 3: потери 0.590 | отложенные NMT 0.829 | шагов с NaN/inf 0 | 370 с
  выбор lr 0.0005 запуск 0 | эпоха 4: потери 0.537 | отложенные NMT 0.891 | шагов с NaN/inf 0 | 491 с
  выбор lr 0.0005 запуск 0 | эпоха 5: потери 0.485 | отложенные NMT 0.885 | шагов с NaN/inf 0 | 613 с
  выбор lr 0.0005 запуск 0 | эпоха 6: потери 0.455 | отложенные NMT 0.891 | шагов с NaN/inf 0 | 735 с
  выбор lr 0.0005 запуск 0: остановка — время, лучшая эпоха 6
lr 0.0005: лучшая эпоха 6, отложенные NMT 0.891, 12.3 мин
выбрана скорость: 0.0005
eegdino запуск 0: эпоха 6 | NMT eval 0.879 | Турция 0.906 | медиана p здоровые/эпилепсия 0.017/0.305 | разных значений 119/121 | 0.8 мин
  dino lr 0.0005 запуск 1 | эпоха 1: потери 0.674 | отложенные NMT 0.749 | шагов с NaN/inf 0 | 121 с
  dino lr 0.0005 запуск 1 | эпоха 2: потери 0.648 | отложенные NMT 0.757 | шагов с NaN/inf 0 | 242 с
  dino lr 0.0005 запуск 1 | эпоха 3: потери 0.614 | отложенные NMT 0.757 | шагов с NaN/inf 0 | 363 с
  dino lr 0.0005 запуск 1 | эпоха 4: потери 0.580 | отложенные NMT 0.827 | шагов с NaN/inf 0 | 484 с
  dino lr 0.0005 запуск 1 | эпоха 5: потери 0.544 | отложенные NMT 0.869 | шагов с NaN/inf 0 | 606 с
  dino lr 0.0005 запуск 1 | эпоха 6: потери 0.499 | отложенные NMT 0.887 | шагов с NaN/inf 0 | 727 с
  dino lr 0.0005 запуск 1 | эпоха 7: потери 0.472 | отложенные NMT 0.883 | шагов с NaN/inf 0 | 849 с
  dino lr 0.0005 запуск 1 | эпоха 8: потери 0.438 | отложенные NMT 0.877 | шагов с NaN/inf 0 | 970 с
  dino lr 0.0005 запуск 1: остановка — плато, лучшая эпоха 6
eegdino запуск 1: эпоха 6 | NMT eval 0.859 | Турция 0.776 | медиана p здоровые/эпилепсия 0.050/0.175 | разных значений 119/121 | 17.0 мин
ПРОПУСК запуск 2: не укладывается в 58 мин
всё обучение: 52.5 мин
  model  lr  seed  best_epoch  stop   val  nan_steps  nmt_eval  turkey  turkey_med_p0  turkey_med_p1  n_unique_tr
eegdino 0.0     0           6 время 0.891          0     0.879   0.906          0.017          0.305          119
eegdino 0.0     1           6 плато 0.887          0     0.859   0.776          0.050          0.175          119
                                   модель n          NMT eval                      Турция
          EEG-DINO-M дообученная, запуски 2     0.869 ± 0.014 0.841 ± 0.092 (0.91 / 0.78)
EEG-DINO-M дообученная, ансамбль запусков   0.877 (0.83-0.92)           0.865 (0.80-0.92)
Связь с амплитудой записи на Турции (Спирмен): 0.25
  модель                 порог  значение порога    набор   n         accuracy сбаланс. accuracy        precision recall (чувств.)    специфичность
EEG-DINO                  Юден            0.327 NMT eval 185 0.79 (0.73-0.84)  0.79 (0.73-0.84) 0.76 (0.67-0.84) 0.83 (0.75-0.91) 0.75 (0.66-0.83)
EEG-DINO                  Юден            0.327   Турция 121 0.73 (0.65-0.80)  0.67 (0.61-0.74) 0.95 (0.83-1.00) 0.36 (0.23-0.50) 0.99 (0.95-1.00)
EEG-DINO чувствительность 80 %            0.411 NMT eval 185 0.78 (0.72-0.84)  0.78 (0.72-0.84) 0.78 (0.69-0.87) 0.77 (0.68-0.85) 0.80 (0.71-0.88)
EEG-DINO чувствительность 80 %            0.411   Турция 121 0.69 (0.62-0.78)  0.63 (0.57-0.70) 1.00 (1.00-1.00) 0.26 (0.15-0.39) 1.00 (1.00-1.00)
всё время блокнота: 70.9 мин
```

## 6а. Третий запуск (seed 2) — отдельный блокнот Kaggle, пересчёт итогов по csv (локально, без GPU)

Seed 2 не уложился в бюджет основного расчёта и досчитан отдельно (`zakharsann/h1-eegdino-seed2`): те же ячейки подготовки и обучения, скорость 5·10⁻⁴ (выбрана выше), до 10 эпох с остановкой на плато. Ниже его вероятности добавляются в `eegdino_finetune_per_person.csv`, после чего пересчитываются ансамбль из 3 запусков, сводка и пороговые метрики — тем же кодом, что в разделе 5. Порог — снова по отложенным 15 % NMT train (среднее вероятностей 3 запусков).

In [ ]:
# Добавление seed 2 и пересчёт ансамбля, сводки и пороговых метрик по сохранённым csv (без GPU)
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, roc_curve
from scipy.stats import rankdata, spearmanr

RES = "../results"
pp = pd.read_csv(f"{RES}/eegdino_finetune_per_person.csv")
s2 = pd.read_csv(f"{RES}/eegdino_seed2_per_person.csv")
if "eegdino_seed2" in pp.columns:
    pp = pp.drop(columns="eegdino_seed2")
pp = pp.merge(s2, on=["dataset", "id"], how="left", suffixes=("", "_s2"))
assert pp.eegdino_seed2.notna().all() and (pp.y == pp.y_s2).all() and len(pp) == len(s2)
pp = pp.drop(columns="y_s2")
seeds = [c for c in pp.columns if c.startswith("eegdino_seed")]
rank = lambda s: rankdata(s) / len(s)
for ds in ["nmt_eval", "turkey"]:
    i = pp.dataset == ds
    pp.loc[i, "eegdino_ens_rank"] = np.mean([rank(pp.loc[i, c]) for c in seeds], 0)
pp.to_csv(f"{RES}/eegdino_finetune_per_person.csv", index=False)

runs = pd.read_csv(f"{RES}/eegdino_finetune_runs.csv")
runs = pd.concat([runs[runs.seed != 2], pd.read_csv(f"{RES}/eegdino_seed2_run.csv")], ignore_index=True)
runs.to_csv(f"{RES}/eegdino_finetune_runs.csv", index=False)
print(runs.drop(columns="hist").round(3).to_string(index=False))

def boot_ci(yt, s, n=2000, seed=0):
    r = np.random.default_rng(seed); v = []
    for _ in range(n):
        i = r.integers(0, len(yt), len(yt))
        if len(set(yt[i])) == 2: v.append(roc_auc_score(yt[i], s[i]))
    return np.percentile(v, [2.5, 97.5])
def fmt(y, s):
    lo, hi = boot_ci(y, s); return f"{roc_auc_score(y, s):.3f} ({lo:.2f}-{hi:.2f})"
E, T, V = (pp[pp.dataset == d] for d in ["nmt_eval", "turkey", "nmt_val"])
y_ev, y_tr, y_val = E.y.to_numpy(), T.y.to_numpy(), V.y.to_numpy()
summary = pd.DataFrame([
    {"модель": "EEG-DINO-M дообученная, запуски", "n": len(runs), "NMT eval": f"{runs.nmt_eval.mean():.3f} ± {runs.nmt_eval.std(ddof=1):.3f}",
     "Турция": f"{runs.turkey.mean():.3f} ± {runs.turkey.std(ddof=1):.3f} ({' / '.join(f'{v:.2f}' for v in runs.turkey)})"},
    {"модель": "EEG-DINO-M дообученная, ансамбль запусков", "n": "", "NMT eval": fmt(y_ev, E.eegdino_ens_rank.to_numpy()), "Турция": fmt(y_tr, T.eegdino_ens_rank.to_numpy())}])
summary.to_csv(f"{RES}/eegdino_finetune_summary.csv", index=False)
print(summary.to_string(index=False))
print("Связь с амплитудой записи на Турции (Спирмен):", round(spearmanr(T.eegdino_ens_rank, T.amp).correlation, 2))

# пороговые метрики: оценка — среднее вероятностей запусков, порог по отложенным 15 % NMT train (как в разделе 5)
p_val, p_ev, p_tr = (D[seeds].mean(1).to_numpy() for D in (V, E, T))
fpr, tpr, thr = roc_curve(y_val, p_val)
THR = {"Юден": float(thr[np.argmax(tpr - fpr)]), "чувствительность 80 %": float(thr[np.argmax(tpr >= 0.8)])}
def metr(y, pr):
    tp, tn = ((pr == 1) & (y == 1)).sum(), ((pr == 0) & (y == 0)).sum(); fp, fn = ((pr == 1) & (y == 0)).sum(), ((pr == 0) & (y == 1)).sum()
    rec = tp / max(tp + fn, 1); spec = tn / max(tn + fp, 1)
    return dict(acc=(tp + tn) / len(y), bacc=(rec + spec) / 2, prec=tp / (tp + fp) if tp + fp else np.nan, rec=rec, spec=spec)
def metr_ci(y, s, t, n=2000, seed=0):
    pr = (s >= t).astype(int); m0 = metr(y, pr); r = np.random.default_rng(seed); B = []
    for _ in range(n):
        i = r.integers(0, len(y), len(y)); B.append(metr(y[i], pr[i]))
    return {k: f"{m0[k]:.2f} ({np.nanpercentile([b[k] for b in B], 2.5):.2f}-{np.nanpercentile([b[k] for b in B], 97.5):.2f})" for k in m0}
rows = []
for name, t in THR.items():
    for ds, y, s in [("NMT eval", y_ev, p_ev), ("Турция", y_tr, p_tr)]:
        m = metr_ci(y, s, t)
        rows.append({"модель": "EEG-DINO", "порог": name, "значение порога": round(t, 3), "набор": ds,
                     "примечание": "порог: отложенные 15 % NMT train" + ("" if ds == "NMT eval" else "; перенесён"), "n": len(y),
                     "accuracy": m["acc"], "сбаланс. accuracy": m["bacc"], "precision": m["prec"], "recall (чувств.)": m["rec"], "специфичность": m["spec"]})
thr_tab = pd.DataFrame(rows)
thr_tab.to_csv(f"{RES}/eegdino_threshold_metrics.csv", index=False)
print(thr_tab.drop(columns=["примечание"]).to_string(index=False))

Вывод Kaggle (seed 2, без строк о подкачке и служебных предупреждений):

```
в файле весов: {'crops_projector': 8, 'dino_loss': 1, 'ibot_patch_loss': 1, 'mask_projector': 8, 'student': 227, 'teacher': 227}
EEG-DINO-M: кодировщик 227 тензоров загружено строго, без обучения остаются только тензоры головы (10): ['channel_linear', 'classifier', 'full_linear']
параметров: 34445121 | из них кодировщик: 33755456
устройство: Tesla T4
label  abnormal  normal
loc                    
eval         90      95
train       325    1907
NMT прочитано 400/2417 145 с
NMT прочитано 800/2417 289 с
NMT прочитано 1200/2417 431 с
NMT прочитано 1600/2417 585 с
NMT прочитано 2000/2417 729 с
NMT прочитано 2400/2417 885 с
NMT прочитано 2417/2417 892 с
label  abnormal  normal
loc                    
eval         90      95
train       325    1904
окон по 10 с: train 17775 | eval 10406
Турция прочитано за 143 с | эпилепсия 50, здоровые 71 | окон на запись: медиана 60
эталон перекоса: СКО сигнала записи как оценка, ROC AUC = 0.764
окон для обучения: 15103 (патология 0.14) | записей на выбор: 335
  dino lr 0.0005 запуск 2 | эпоха 1: потери 0.662 | отложенные NMT 0.740 | шагов с NaN/inf 0 | 115 с
  dino lr 0.0005 запуск 2 | эпоха 2: потери 0.633 | отложенные NMT 0.760 | шагов с NaN/inf 0 | 227 с
  dino lr 0.0005 запуск 2 | эпоха 3: потери 0.623 | отложенные NMT 0.770 | шагов с NaN/inf 0 | 339 с
  dino lr 0.0005 запуск 2 | эпоха 4: потери 0.597 | отложенные NMT 0.839 | шагов с NaN/inf 0 | 451 с
  dino lr 0.0005 запуск 2 | эпоха 5: потери 0.561 | отложенные NMT 0.857 | шагов с NaN/inf 0 | 564 с
  dino lr 0.0005 запуск 2 | эпоха 6: потери 0.530 | отложенные NMT 0.872 | шагов с NaN/inf 0 | 676 с
  dino lr 0.0005 запуск 2 | эпоха 7: потери 0.482 | отложенные NMT 0.871 | шагов с NaN/inf 0 | 788 с
  dino lr 0.0005 запуск 2 | эпоха 8: потери 0.450 | отложенные NMT 0.873 | шагов с NaN/inf 0 | 901 с
  dino lr 0.0005 запуск 2 | эпоха 9: потери 0.422 | отложенные NMT 0.881 | шагов с NaN/inf 0 | 1013 с
  dino lr 0.0005 запуск 2 | эпоха 10: потери 0.413 | отложенные NMT 0.877 | шагов с NaN/inf 0 | 1125 с
  dino lr 0.0005 запуск 2: остановка — макс. эпох, лучшая эпоха 9
eegdino запуск 2: эпоха 9 (макс. эпох) | отложенные 0.881 | NMT eval 0.867 | Турция 0.816 | медиана p здоровые/эпилепсия 0.008/0.057 | разных значений 119/121 | 19.4 мин
всё время блокнота: 37.7 мин
```

Вывод пересчёта (локально):

```
model  lr  seed  best_epoch       stop   val  nan_steps  nmt_eval  turkey  turkey_med_p0  turkey_med_p1  n_unique_tr
eegdino 0.0     0           6      время 0.891          0     0.879   0.906          0.017          0.305          119
eegdino 0.0     1           6      плато 0.887          0     0.859   0.776          0.050          0.175          119
eegdino 0.0     2           9 макс. эпох 0.881          0     0.867   0.816          0.008          0.057          119
                                   модель n          NMT eval                             Турция
          EEG-DINO-M дообученная, запуски 3     0.868 ± 0.010 0.833 ± 0.066 (0.91 / 0.78 / 0.82)
EEG-DINO-M дообученная, ансамбль запусков   0.875 (0.82-0.92)                  0.856 (0.79-0.91)
Связь с амплитудой записи на Турции (Спирмен): 0.24
  модель                 порог  значение порога    набор   n         accuracy сбаланс. accuracy        precision recall (чувств.)    специфичность
EEG-DINO                  Юден            0.324 NMT eval 185 0.78 (0.72-0.84)  0.79 (0.72-0.84) 0.75 (0.66-0.83) 0.83 (0.75-0.91) 0.74 (0.64-0.82)
EEG-DINO                  Юден            0.324   Турция 121 0.70 (0.63-0.79)  0.64 (0.58-0.71) 1.00 (1.00-1.00) 0.28 (0.16-0.42) 1.00 (1.00-1.00)
EEG-DINO чувствительность 80 %            0.433 NMT eval 185 0.79 (0.74-0.85)  0.79 (0.73-0.85) 0.80 (0.71-0.89) 0.77 (0.68-0.85) 0.82 (0.74-0.90)
EEG-DINO чувствительность 80 %            0.433   Турция 121 0.64 (0.56-0.73)  0.57 (0.52-0.62) 1.00 (1.00-1.00) 0.14 (0.05-0.25) 1.00 (1.00-1.00)
```

## 7. Сравнение с другими моделями, ансамбль с REVE, калибровка на месте (локально, без GPU, по сохранённым предсказаниям)

Запускать из папки блокнота: читает `results/eegdino_finetune_per_person.csv`, `results/biot_ensembles_per_person.csv` (предсказания REVE, CBraMod, BIOT, DeepEpilepsy по людям) и `results/luna_ensembles_per_person.csv` (LUNA). Калибровка — та же функция и те же выборки (seed 0, тот же порядок людей), что в `results/luna_turkey_local_threshold_calibration.csv`; совпадение строк REVE/CBraMod/BIOT/LUNA с той таблицей проверяется.

In [ ]:
# Сравнение EEG-DINO с другими моделями по сохранённым предсказаниям (без GPU): ансамбль с REVE, парные разности, калибровка на месте
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, roc_curve
from scipy.stats import rankdata, spearmanr

RES = "../results"
dn = pd.read_csv(f"{RES}/eegdino_finetune_per_person.csv")
ot = pd.read_csv(f"{RES}/biot_ensembles_per_person.csv")
lu = pd.read_csv(f"{RES}/luna_ensembles_per_person.csv")
dn_cols = [c for c in dn.columns if c.startswith("eegdino_seed")]
m = dn[dn.dataset.isin(["nmt_eval", "turkey"])].merge(
    ot[["dataset", "id", "y", "reve_ens_rank", "cb_ens_rank", "de_ens_rank", "biot_ens_rank"]], on=["dataset", "id"], suffixes=("", "_o")).merge(
    lu[["dataset", "id", "y", "luna_rank"]], on=["dataset", "id"], suffixes=("", "_l"))
assert (m.y == m.y_o).all() and (m.y == m.y_l).all()
print({d: int((m.dataset == d).sum()) for d in ["nmt_eval", "turkey"]}, "общих записей")

rk = lambda s: rankdata(s) / len(s)
SC = {}
for ds in ["nmt_eval", "turkey"]:
    d = m[m.dataset == ds]
    s = {"EEG-DINO": np.mean([rk(d[c]) for c in dn_cols], 0), "REVE": rk(d.reve_ens_rank), "CBraMod": rk(d.cb_ens_rank),
         "BIOT": rk(d.biot_ens_rank), "LUNA": rk(d.luna_rank), "DeepEpilepsy": rk(d.de_ens_rank)}
    s["EEG-DINO + REVE"] = (s["EEG-DINO"] + s["REVE"]) / 2
    s["EEG-DINO + REVE + CBraMod"] = (s["EEG-DINO"] + s["REVE"] + s["CBraMod"]) / 3
    SC[ds] = (d.y.to_numpy(), s, d)

def boot(y, f, n=2000, seed=0):
    r = np.random.default_rng(seed); v = []
    for _ in range(n):
        i = r.integers(0, len(y), len(y))
        if len(set(y[i])) == 2: v.append(f(i))
    return np.percentile(v, [2.5, 97.5])

rows = []
for name in SC["turkey"][1]:
    row = {"модель / ансамбль (средние ранги)": name}
    for ds, lab in [("nmt_eval", "NMT eval"), ("turkey", "Турция")]:
        y, s, _ = SC[ds]; lo, hi = boot(y, lambda i: roc_auc_score(y[i], s[name][i]))
        row[lab] = f"{roc_auc_score(y, s[name]):.3f} ({lo:.2f}-{hi:.2f})"
    rows.append(row)
summ = pd.DataFrame(rows)
print(summ.to_string(index=False))

print("\nПарный бутстреп разности ROC AUC (95% ДИ):")
diffs = []
for a, b in [("EEG-DINO", "REVE"), ("EEG-DINO", "CBraMod"), ("EEG-DINO", "BIOT"), ("EEG-DINO", "LUNA"), ("EEG-DINO", "DeepEpilepsy"),
             ("EEG-DINO + REVE", "REVE"), ("EEG-DINO + REVE", "EEG-DINO")]:
    for ds, lab in [("nmt_eval", "NMT eval"), ("turkey", "Турция")]:
        y, s, _ = SC[ds]
        f = lambda i: roc_auc_score(y[i], s[a][i]) - roc_auc_score(y[i], s[b][i])
        lo, hi = boot(y, f); dv = f(np.arange(len(y)))
        diffs.append({"разность": f"{a} − {b}", "набор": lab, "ΔROC AUC": round(dv, 3), "ДИ": f"{lo:+.3f}…{hi:+.3f}"})
        print(f"  {a} − {b} | {lab}: {dv:+.3f} ({lo:+.3f}…{hi:+.3f})")
y, s, d = SC["turkey"]
sp = {"амплитуда": d.amp, **{k: s[k] for k in ["REVE", "CBraMod", "BIOT", "LUNA", "DeepEpilepsy"]}}
print("\nСпирмен на Турции, EEG-DINO с:", " | ".join(f"{k} {spearmanr(s['EEG-DINO'], v).correlation:.2f}" for k, v in sp.items()))
print("одна амплитуда как оценка, ROC AUC:", round(roc_auc_score(y, d.amp), 3))

# --- калибровка на месте: порог по k размеченным людям Турции (поровну классов, индекс Юдена), проверка на остальных; 500 повторов, одни и те же выборки для всех моделей ---
def calib(y, sc, k, reps=500, seed=0):
    r = np.random.default_rng(seed); i1, i0 = np.where(y == 1)[0], np.where(y == 0)[0]; M = {n: [] for n in sc}
    for _ in range(reps):
        c = np.concatenate([r.choice(i1, k // 2, replace=False), r.choice(i0, k // 2, replace=False)])
        rest = np.setdiff1d(np.arange(len(y)), c); yy = y[rest]
        for n, s in sc.items():
            fpr, tpr, th = roc_curve(y[c], s[c]); pr = s[rest] >= th[np.argmax(tpr - fpr)]
            tp = (pr & (yy == 1)).sum(); tn = (~pr & (yy == 0)).sum(); fp = (pr & (yy == 0)).sum(); fn = (~pr & (yy == 1)).sum()
            rec, spec = tp / (tp + fn), tn / (tn + fp)
            M[n].append(dict(rec=rec, spec=spec, bacc=(rec + spec) / 2, prec=tp / (tp + fp) if tp + fp else np.nan, acc=(tp + tn) / len(yy)))
    out = []
    for n in sc:
        D = pd.DataFrame(M[n]); f = lambda c: f"{D[c].mean():.2f} ({np.nanpercentile(D[c], 2.5):.2f}–{np.nanpercentile(D[c], 97.5):.2f})"
        out.append({"калибровка": f"{k} чел.", "модель": n, "recall": f("rec"), "специфичность": f("spec"), "сбаланс. accuracy": f("bacc"),
                    "precision": f("prec"), "accuracy": f("acc")})
    return out
cal_models = ["EEG-DINO", "REVE", "CBraMod", "BIOT", "LUNA", "DeepEpilepsy", "EEG-DINO + REVE"]
cal = pd.DataFrame([row for k in [10, 20, 30] for row in calib(y, {n: s[n] for n in cal_models}, k)])
print("\nКалибровка на месте (среднее по 500 повторам, 2,5–97,5 % повторов):")
print(cal.to_string(index=False))

# проверка: те же выборки, что в таблице LUNA — строки общих моделей должны совпасть
old = pd.read_csv(f"{RES}/luna_turkey_local_threshold_calibration.csv")
chk = cal.merge(old, on=["калибровка", "модель"], suffixes=("", "_old"))
cols = ["recall", "специфичность", "сбаланс. accuracy", "precision", "accuracy"]
same = all((chk[c] == chk[c + "_old"]).all() for c in cols)
print(f"\nсовпадение с luna_turkey_local_threshold_calibration.csv по {len(chk)} общим строкам ({sorted(set(chk['модель']))}):", same)
assert same

summ.to_csv(f"{RES}/eegdino_ensembles_summary.csv", index=False)
pd.DataFrame(diffs).to_csv(f"{RES}/eegdino_paired_differences.csv", index=False)
cal.to_csv(f"{RES}/eegdino_turkey_local_threshold_calibration.csv", index=False)
pp = []
for ds in ["nmt_eval", "turkey"]:
    yy, s, d = SC[ds]; t = d[["dataset", "id", "y"]].copy()
    for n, v in s.items(): t[n.replace(" + ", "_").replace(" ", "").replace("-", "").lower() + "_rank"] = v
    pp.append(t)
pd.concat(pp).to_csv(f"{RES}/eegdino_ensembles_per_person.csv", index=False)

Вывод (выполнено локально, 3 запуска EEG-DINO):

```
{'nmt_eval': 184, 'turkey': 121} общих записей
модель / ансамбль (средние ранги)          NMT eval            Турция
                         EEG-DINO 0.878 (0.83-0.92) 0.856 (0.79-0.91)
                             REVE 0.879 (0.83-0.92) 0.874 (0.80-0.93)
                          CBraMod 0.845 (0.79-0.90) 0.820 (0.74-0.88)
                             BIOT 0.798 (0.73-0.86) 0.821 (0.73-0.89)
                             LUNA 0.858 (0.80-0.91) 0.795 (0.71-0.87)
                     DeepEpilepsy 0.900 (0.85-0.94) 0.682 (0.59-0.77)
                  EEG-DINO + REVE 0.893 (0.84-0.93) 0.896 (0.83-0.95)
        EEG-DINO + REVE + CBraMod 0.885 (0.83-0.93) 0.885 (0.82-0.94)

Парный бутстреп разности ROC AUC (95% ДИ):
  EEG-DINO − REVE | NMT eval: -0.001 (-0.040…+0.038)
  EEG-DINO − REVE | Турция: -0.018 (-0.080…+0.044)
  EEG-DINO − CBraMod | NMT eval: +0.033 (-0.003…+0.072)
  EEG-DINO − CBraMod | Турция: +0.036 (-0.029…+0.105)
  EEG-DINO − BIOT | NMT eval: +0.081 (+0.036…+0.131)
  EEG-DINO − BIOT | Турция: +0.035 (-0.042…+0.112)
  EEG-DINO − LUNA | NMT eval: +0.021 (-0.019…+0.063)
  EEG-DINO − LUNA | Турция: +0.061 (-0.018…+0.147)
  EEG-DINO − DeepEpilepsy | NMT eval: -0.021 (-0.062…+0.017)
  EEG-DINO − DeepEpilepsy | Турция: +0.175 (+0.073…+0.268)
  EEG-DINO + REVE − REVE | NMT eval: +0.014 (-0.005…+0.035)
  EEG-DINO + REVE − REVE | Турция: +0.022 (-0.004…+0.054)
  EEG-DINO + REVE − EEG-DINO | NMT eval: +0.014 (-0.006…+0.038)
  EEG-DINO + REVE − EEG-DINO | Турция: +0.040 (+0.002…+0.083)

Спирмен на Турции, EEG-DINO с: амплитуда 0.24 | REVE 0.68 | CBraMod 0.70 | BIOT 0.63 | LUNA 0.63 | DeepEpilepsy 0.39
одна амплитуда как оценка, ROC AUC: 0.764

Калибровка на месте (среднее по 500 повторам, 2,5–97,5 % повторов):
калибровка          модель           recall    специфичность сбаланс. accuracy        precision         accuracy
   10 чел.        EEG-DINO 0.64 (0.22–0.96) 0.80 (0.56–1.00)  0.72 (0.61–0.78) 0.73 (0.59–1.00) 0.74 (0.68–0.77)
   10 чел.            REVE 0.62 (0.24–0.91) 0.88 (0.50–1.00)  0.75 (0.62–0.81) 0.85 (0.55–1.00) 0.78 (0.66–0.85)
   10 чел.         CBraMod 0.61 (0.18–0.96) 0.76 (0.36–1.00)  0.69 (0.59–0.74) 0.70 (0.51–1.00) 0.70 (0.60–0.74)
   10 чел.            BIOT 0.59 (0.14–0.92) 0.83 (0.36–0.98)  0.71 (0.56–0.77) 0.77 (0.49–0.92) 0.73 (0.59–0.80)
   10 чел.            LUNA 0.55 (0.13–0.89) 0.82 (0.36–1.00)  0.69 (0.56–0.73) 0.77 (0.49–1.00) 0.71 (0.59–0.77)
   10 чел.    DeepEpilepsy 0.54 (0.02–0.94) 0.65 (0.22–1.00)  0.59 (0.51–0.65) 0.55 (0.45–0.80) 0.60 (0.52–0.64)
   10 чел. EEG-DINO + REVE 0.65 (0.27–0.93) 0.90 (0.55–1.00)  0.77 (0.63–0.83) 0.86 (0.58–1.00) 0.80 (0.67–0.86)
   20 чел.        EEG-DINO 0.72 (0.30–0.96) 0.75 (0.54–0.98)  0.74 (0.63–0.78) 0.68 (0.57–0.92) 0.74 (0.69–0.78)
   20 чел.            REVE 0.65 (0.38–0.90) 0.88 (0.49–1.00)  0.77 (0.66–0.82) 0.84 (0.54–1.00) 0.79 (0.65–0.86)
   20 чел.         CBraMod 0.66 (0.25–0.95) 0.73 (0.39–1.00)  0.69 (0.61–0.74) 0.66 (0.51–1.00) 0.70 (0.61–0.74)
   20 чел.            BIOT 0.64 (0.34–0.88) 0.82 (0.43–0.98)  0.73 (0.63–0.78) 0.75 (0.50–0.93) 0.75 (0.60–0.81)
   20 чел.            LUNA 0.56 (0.25–0.93) 0.82 (0.34–1.00)  0.69 (0.61–0.74) 0.76 (0.48–1.00) 0.72 (0.57–0.78)
   20 чел.    DeepEpilepsy 0.64 (0.10–0.95) 0.57 (0.25–0.97)  0.60 (0.52–0.66) 0.51 (0.44–0.71) 0.60 (0.52–0.65)
   20 чел. EEG-DINO + REVE 0.70 (0.42–0.90) 0.89 (0.57–0.98)  0.79 (0.69–0.84) 0.84 (0.59–0.97) 0.81 (0.71–0.87)
   30 чел.        EEG-DINO 0.75 (0.40–0.97) 0.73 (0.54–0.95)  0.74 (0.66–0.79) 0.65 (0.56–0.82) 0.74 (0.68–0.79)
   30 чел.            REVE 0.67 (0.43–0.91) 0.87 (0.46–1.00)  0.77 (0.68–0.83) 0.82 (0.51–1.00) 0.79 (0.63–0.87)
   30 чел.         CBraMod 0.68 (0.26–0.94) 0.72 (0.51–1.00)  0.70 (0.62–0.76) 0.64 (0.53–1.00) 0.70 (0.65–0.76)
   30 чел.            BIOT 0.64 (0.40–0.86) 0.82 (0.62–0.98)  0.73 (0.66–0.79) 0.74 (0.57–0.93) 0.75 (0.69–0.81)
   30 чел.            LUNA 0.56 (0.29–0.91) 0.83 (0.36–1.00)  0.69 (0.63–0.75) 0.76 (0.47–1.00) 0.73 (0.57–0.80)
   30 чел.    DeepEpilepsy 0.66 (0.14–0.94) 0.55 (0.23–0.95)  0.61 (0.52–0.67) 0.49 (0.42–0.65) 0.59 (0.50–0.65)
   30 чел. EEG-DINO + REVE 0.72 (0.49–0.91) 0.87 (0.65–0.98)  0.80 (0.72–0.84) 0.81 (0.61–0.96) 0.81 (0.73–0.87)

совпадение с luna_turkey_local_threshold_calibration.csv по 15 общим строкам (['BIOT', 'CBraMod', 'DeepEpilepsy', 'LUNA', 'REVE']): True
```

## 8. Итоги (3 запуска)

### ROC AUC (95% ДИ, бутстреп по людям; ранги пересчитаны на общих записях: NMT eval 184 + Турция 121)

| Модель / ансамбль (средние ранги) | NMT eval | Турция |
|---|---|---|
| **EEG-DINO-M (3 запуска)** | 0,878 (0,83–0,92) | 0,856 (0,79–0,91) |
| REVE | 0,879 (0,83–0,92) | 0,874 (0,80–0,93) |
| CBraMod | 0,845 (0,79–0,90) | 0,820 (0,74–0,88) |
| BIOT | 0,798 (0,73–0,86) | 0,821 (0,73–0,89) |
| LUNA | 0,858 (0,80–0,91) | 0,795 (0,71–0,87) |
| DeepEpilepsy | **0,900 (0,85–0,94)** | 0,682 (0,59–0,77) |
| **EEG-DINO + REVE** | 0,893 (0,84–0,93) | **0,896 (0,83–0,95)** |
| EEG-DINO + REVE + CBraMod | 0,885 (0,83–0,93) | 0,885 (0,82–0,94) |

Запуски EEG-DINO (seed 0 / 1 / 2): NMT eval 0,88 / 0,86 / 0,87 (0,868 ± 0,010), **Турция 0,91 / 0,78 / 0,82** (0,833 ± 0,066). Лучшие эпохи 6 / 6 / 9; остановка: по времени / плато / максимум 10 эпох. На всех 185 записях NMT eval ансамбль 0,875 (0,82–0,92). С двумя запусками было 0,865 на Турции; третий запуск слабее первого, и ансамбль опустился на 0,01.

Выбор скорости по отложенным 15 % NMT train (лучший ROC AUC / эпоха / остановка): 5·10⁻⁵ — 0,872 / 3 / плато; 1,5·10⁻⁴ — 0,877 / 4 / плато; **5·10⁻⁴ (значение авторов) — 0,891 / 6 / время** → выбрана 5·10⁻⁴. Это край сетки. С этой скоростью модель учится медленно: в запуске 2 отложенный AUC рос до 9-й эпохи (0,881), на 10-й — 0,877.

Парный бутстреп разности ROC AUC (95% ДИ):
- EEG-DINO − REVE: Турция −0,018 (−0,080…+0,044), NMT eval −0,001 — **равны**.
- EEG-DINO − CBraMod: Турция +0,036 (−0,029…+0,105), NMT eval +0,033 (−0,003…+0,072) — не значимо.
- EEG-DINO − BIOT: Турция +0,035 (не значимо); NMT eval +0,081 (+0,036…+0,131).
- EEG-DINO − LUNA: Турция +0,061 (−0,018…+0,147), NMT eval +0,021 — не значимо.
- EEG-DINO − DeepEpilepsy: Турция +0,175 (+0,073…+0,268); NMT eval −0,021 (не значимо).
- **EEG-DINO + REVE − REVE: Турция +0,022 (−0,004…+0,054)**, NMT eval +0,014 (−0,005…+0,035) — не значимо.
- EEG-DINO + REVE − EEG-DINO: Турция +0,040 (+0,002…+0,083).

Связь с амплитудой записи на Турции (Спирмен): +0,24 (REVE +0,31, BIOT +0,25, LUNA +0,19, CBraMod +0,12; одна амплитуда даёт AUC 0,76). Согласие с другими моделями: CBraMod 0,70, REVE 0,68, BIOT 0,63, LUNA 0,63, DeepEpilepsy 0,39.

### Пороговые метрики (оценка — среднее вероятностей 3 запусков; порог по отложенным 15 % NMT train, на Турцию перенесён тот же)

| Порог | Набор | Accuracy | Сбаланс. accuracy | Precision | Recall | Специфичность |
|---|---|---|---|---|---|---|
| Юден (0,324) | NMT eval (185) | 0,78 (0,72–0,84) | 0,79 (0,72–0,84) | 0,75 (0,66–0,83) | 0,83 (0,75–0,91) | 0,74 (0,64–0,82) |
| то же, перенесён | Турция (121) | 0,70 (0,63–0,79) | 0,64 (0,58–0,71) | 1,00 | 0,28 (0,16–0,42) | 1,00 |
| чувств. 80 % (0,433) | NMT eval | 0,79 (0,74–0,85) | 0,79 (0,73–0,85) | 0,80 (0,71–0,89) | 0,77 (0,68–0,85) | 0,82 (0,74–0,90) |
| то же, перенесён | Турция | 0,64 (0,56–0,73) | 0,57 (0,52–0,62) | 1,00 | 0,14 (0,05–0,25) | 1,00 |

### Калибровка на месте (Турция): порог по k размеченным людям (поровну классов, Юден), проверка на остальных, 500 повторов (среднее, 2,5–97,5 % повторов)

Те же выборки и тот же код, что в `luna_turkey_local_threshold_calibration.csv`: строки REVE, CBraMod, BIOT, LUNA, DeepEpilepsy совпали с той таблицей полностью.

| k | Модель | Сбаланс. accuracy | Recall | Специфичность | Precision | Accuracy |
|---|---|---|---|---|---|---|
| 10 | EEG-DINO | 0,72 (0,61–0,78) | 0,64 | 0,80 | 0,73 | 0,74 |
| 10 | REVE | 0,75 (0,62–0,81) | 0,62 | 0,88 | 0,85 | 0,78 |
| 10 | EEG-DINO + REVE | 0,77 (0,63–0,83) | 0,65 | 0,90 | 0,86 | 0,80 |
| 20 | **EEG-DINO** | **0,74 (0,63–0,78)** | 0,72 (0,30–0,96) | 0,75 (0,54–0,98) | 0,68 | 0,74 |
| 20 | REVE | 0,77 (0,66–0,82) | 0,65 | 0,88 | 0,84 | 0,79 |
| 20 | CBraMod / BIOT / LUNA / DeepEpilepsy | 0,69 / 0,73 / 0,69 / 0,60 | | | | |
| 20 | **EEG-DINO + REVE** | **0,79 (0,69–0,84)** | 0,70 (0,42–0,90) | 0,89 (0,57–0,98) | 0,84 | 0,81 |
| 30 | EEG-DINO | 0,74 (0,66–0,79) | 0,75 | 0,73 | 0,65 | 0,74 |
| 30 | REVE | 0,77 (0,68–0,83) | 0,67 | 0,87 | 0,82 | 0,79 |
| 30 | EEG-DINO + REVE | 0,80 (0,72–0,84) | 0,72 | 0,87 | 0,81 | 0,81 |

Полная таблица — `results/eegdino_turkey_local_threshold_calibration.csv`.

## 9. Выводы

1. **EEG-DINO-M переносится наравне с REVE**: Турция 0,856 (0,79–0,91) против 0,874, разность −0,02 (−0,08…+0,04); на NMT eval 0,878 — тоже наравне. Лучше CBraMod, BIOT и LUNA на 0,04–0,06, но не значимо; DeepEpilepsy обходит на Турции значимо (+0,18).
2. **Ансамбль EEG-DINO + REVE — лучший на Турции**: 0,896 (0,83–0,95). Прирост к одной REVE +0,022 (ДИ −0,004…+0,054) не значим. При калибровке порога на 20 людях сбалансированная accuracy 0,79 (REVE 0,77, EEG-DINO 0,74) — лучшая в таблице, но интервалы перекрываются. EEG-DINO и REVE согласованы так же, как LUNA и REVE (Спирмен 0,68 и 0,69). Но ансамбль с LUNA прироста не давал, а с EEG-DINO даёт.
3. **Нестабильность между запусками**: на Турции 0,91 / 0,78 / 0,82 при близком отложенном AUC на NMT (0,891 / 0,887 / 0,881). Отбор эпохи по NMT не гарантирует перенос. У REVE разброс меньше (0,85–0,87).
4. **Скорость обучения на краю сетки**: выбрана 5·10⁻⁴ (максимум сетки, значение авторов). Отрыв на отложенных небольшой (0,891 против 0,877 при 335 записях). Обучение с ней медленное: лучшие эпохи 6–9 из 10. Большие скорости и больше эпох не проверялись.
5. **Порог с NMT не переносится** (чувствительность 0,28 при специфичности 1,00; у REVE, CBraMod, BIOT, LUNA 0–0,16). Калибровка на месте нужна.
6. **Амплитуда**: Спирмен +0,24 — на уровне BIOT, ниже REVE (+0,31). Окно нормируется к общему СКО, но соотношение каналов и форма сигнала остаются.
7. **Вход и допущения**: 200 Гц, патчи 1 с, мкВ/100 — по коду авторов. Порядок 19 каналов (one-hot вложение по номеру канала) взят из подготовки TUEG у CBraMod, на которую ссылаются авторы; явно авторы его не пишут. Число голов внимания (8) — значение по умолчанию из `run_finetuning.py`, в статье его нет; на форму весов оно не влияет, строгая загрузка его не проверяет.
8. **Утечка для будущего TUEP**: EEG-DINO предобучена только на TUEG (>9000 ч, 1,1 млн кусков по 30 с); исключение пациентов в статье не упомянуто. Пациенты TUEP, вероятно, есть в предобучении — при проверке на TUEP это нужно учитывать, как у CBraMod, REVE и LUNA. Детектор SenuaLab здесь не использовался.
9. **Ограничения**: в Турции 121 человек, ДИ широкие; NMT («патология») и Турция («эпилепсия против здоровых») — разные задачи.

**Затраты:** основной блокнот 70,9 мин на T4 (чтение данных ≈17 мин, обучение 52,5 мин), квота 18,17 → 19,47 ч (≈1,3 ч). Третий запуск — отдельный блокнот 37,7 мин на T4 (чтение ≈17 мин, обучение 19,4 мин) ≈0,6 ч; квота перед ним 20,13 ч, после 20,64 ч (параллельно шёл блокнот EEGPT, так что разность квоты неточна). Всего ≈1,9 ч GPU, недельный потолок 25 ч соблюдён. Короткая проверка на процессоре Kaggle — 9,4 мин, без GPU.

Файлы: `results/eegdino_finetune_{runs,summary,per_person}.csv` (3 запуска; в `per_person` есть вероятности на отложенных 15 %, набор `nmt_val`), `results/eegdino_seed2_{run,per_person}.csv` (вывод третьего запуска), `results/eegdino_lr_choice.csv`, `results/eegdino_threshold_metrics.csv`, `results/eegdino_ensembles_{summary,per_person}.csv`, `results/eegdino_paired_differences.csv`, `results/eegdino_turkey_local_threshold_calibration.csv`. Веса не сохранялись.